# Stage1 原始 FT-900 出口信息 V2

独立来源复核已排除旧口径跨 2012 断点自动延续。冻结基线含原 Stage1 39 输入、发布年龄、分类时期及过渡未知控制；候选仅加同版出口月环比，23 个未知日保留并仅用训练域填补。独立设计批准是首次拟合前硬门。

In [ ]:
from pathlib import Path
import hashlib,json,os
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.core.recorder_sources import load_verified_recorder_objects
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace missing")
    root=root.parent
card=root/"research/stage1_ft900_export_information_v3_20261023"
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized'] and not (card/'run_state.json').exists()
review_path=card/plan['independent_design_review']['file']
assert review_path.is_file(),'independent fixed-design review required before fit'
review=json.loads(review_path.read_text())
assert review['verdict']==plan['independent_design_review']['required_verdict']
assert review['plan_sha256']==sha(plan_path) and review['canonical_source_artifact_sha256']==plan['candidate_source']['mapped_artifact_sha256']
parent=root/plan['parent']['card'];source=root/plan['candidate_source']['card'];spec=plan['candidate_source']
for file,key in [('summary.json','summary_sha256'),('verification_manifest.json','verification_manifest_sha256')]:assert sha(parent/file)==plan['parent'][key]
for file,key in [('summary.json','summary_sha256'),('readback_verification.json','readback_sha256'),('independent_review.json','source_independent_review_sha256')]:assert sha(source/file)==spec[key]
assert sha(root/plan['blocked_prior_design'])==plan['blocked_prior_design_sha256']
artifact=source/'qlib_recorder'/spec['experiment_id']/spec['recorder_id']/'artifacts'/spec['mapped_artifact']
assert sha(artifact)==spec['mapped_artifact_sha256']
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
parent_names=list(plan['parent']['artifact_sha256']);parent_objects,parent_audit=load_verified_recorder_objects(parent,expected_recorder_id=plan['parent']['recorder_id'],artifacts=parent_names,expected_manifest_sha256=plan['parent']['verification_manifest_sha256'],expected_artifact_sha256=plan['parent']['artifact_sha256'])
with R.uri_context(str(source/'qlib_recorder')):
    source_rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_id=spec['experiment_id'])
    mapped=source_rec.load_object(spec['mapped_artifact'])
source_audit={'recorder_id':spec['recorder_id'],'mapped_artifact_sha256':spec['mapped_artifact_sha256'],'independent_source_review_sha256':spec['source_independent_review_sha256']}
base=parent_objects['inputs/features'];frame=parent_objects['inputs/daily_frame'];timing=parent_objects['inputs/timing'];original=parent_objects['models/base_outer_71']
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
mapped=mapped.set_index('signal_date').sort_index()
assert mapped.index.is_unique and len(mapped)==3964
control=pd.DataFrame(index=frame.index)
control['export_release_age_days']=mapped.release_age_days.reindex(frame.index).astype(float)
control['export_post_2012']=mapped.reference_month.reindex(frame.index).ge('2012-01').astype(float)
control['export_transition_unknown']=mapped.transition_quarantined.reindex(frame.index).astype(float)
assert int(mapped.transition_quarantined.sum())==spec['expected_transition_unknown_rows']==23
ip=mapped.mom_pct.reindex(frame.index).rename('export_mom_pct_same_vintage')
forward=frame.soxx_forward_open_return.astype(float)
parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_mask=(frame.index>=pd.Timestamp(plan['cohorts']['train_signal'][0]))&(frame.index<=pd.Timestamp(plan['cohorts']['train_signal'][1]))&(mature<cutoff)&label.notna()
held_mask=(frame.index>=pd.Timestamp(plan['cohorts']['held_reused'][0]))&(frame.index<=pd.Timestamp(plan['cohorts']['held_reused'][1]))&(mature<last)&label.notna()
assert int(train_mask.sum())==plan['cohorts']['expected_mature_train_rows']==3014
assert int(held_mask.sum())==plan['cohorts']['expected_mature_held_rows']==940
assert int(ip.loc[train_mask].isna().sum())==plan['cohorts']['expected_transition_unknown_train_rows']==23
assert int(ip.loc[held_mask].isna().sum())==plan['cohorts']['expected_transition_unknown_held_rows']==0
assert control.loc[train_mask|held_mask].notna().all().all()
controls=list(control.columns);features=base[base_names].copy()
for name in controls:features[name]=control[name]
features[ip.name]=ip
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_mask,'held_eligible':held_mask},index=frame.index)
panel=pd.concat({'feature':features,'label':label.to_frame(),'meta':meta},axis=1)
panel.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(panel),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused'])})
train=dataset.prepare('train',col_set=['feature','label','meta'])
assert len(train)==len(frame.loc[plan['cohorts']['train_signal'][0]:plan['cohorts']['train_signal'][1]])
tr=train.loc[train[('meta','train_eligible')].astype(bool)]
assert len(tr)==3014
preflight={'status':'matched_Stage1_39_plus_canonical_23_unknown_FT900_controls_verified_before_fit','plan_sha256':sha(plan_path),'parent_source_loads':4,'candidate_source_loads':1,'train_mature_rows':len(tr),'held_mature_expected':940,'source_train_unknown':23,'source_held_unknown':0,'historical_first_public_bytes_qualified':False,'classification_continuity_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_scores_read':0}
atomic_json(preflight,card/'preflight.json')
state={'status':'fits_started','plan_sha256':sha(plan_path),'source_object_loads':5,'DatasetH_builds':1,'fit_attempts':[],'fits_done':[],'bootstrap_attempted':False,'native_accounts':0}
atomic_json(state,card/'run_state.json')
yname='soxx_forward_five_open_log_return';ytr=tr[('label',yname)].to_numpy(float)
assert np.isfinite(ytr).all()
base_cols=base_names+controls;candidate_cols=base_cols+[ip.name]
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(state,card/'run_state.json')
    rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,train_features=tr['feature'],train_label=tr['label'],source_audits={'Stage1':parent_audit,'FT900':source_audit})
    models={}
    for name,cols in [('baseline',base_cols),('candidate',candidate_cols)]:
        assert name not in state['fit_attempts'] and len(state['fit_attempts'])<2
        state['fit_attempts'].append(name);atomic_json(state,card/'run_state.json')
        model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
        model.fit(tr['feature'][cols].to_numpy(float),ytr)
        rec.save_objects(artifact_path='models',**{name:model})
        models[name]=model;state['fits_done'].append(name);atomic_json(state,card/'run_state.json')
        rec.log_metrics(completed_fits=len(state['fits_done']))
    artifact_root=tracking/state['experiment_id']/state['recorder_id']/'artifacts'
    seal={'status':'two_fixed_models_sealed_before_held_prepare','plan_sha256':sha(plan_path),'recorder_id':rec.id,'model_sha256':{name:sha(artifact_root/'models'/name) for name in models},'fits_done':list(state['fits_done']),'held_rows_scored':0}
    atomic_json(seal,card/'model_seal.json')
    held=dataset.prepare('held',col_set=['feature','label','meta'])
    he=held.loc[held[('meta','held_eligible')].astype(bool)]
    assert len(he)==plan['cohorts']['expected_mature_held_rows']==940
    yhe=he[('label',yname)].to_numpy(float);assert np.isfinite(yhe).all()
    predictions={name:models[name].predict(he['feature'][cols].to_numpy(float)) for name,cols in [('baseline',base_cols),('candidate',candidate_cols)]}
    dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
    daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
    daily['paired_mse_improvement']=(daily.target-daily.baseline_prediction)**2-(daily.target-daily.candidate_prediction)**2
    assert len(daily)==940 and daily.signal_date.is_monotonic_increasing and np.isfinite(daily.paired_mse_improvement).all()
    halves=[daily.loc[(daily.signal_date>=pd.Timestamp(a))&(daily.signal_date<=pd.Timestamp(b))] for a,b in plan['cohorts']['held_halves']]
    assert sum(len(x) for x in halves)==len(daily) and min(len(x) for x in halves)>300
    b=plan['gate']['paired_stationary_bootstrap'];state['bootstrap_attempted']=True;atomic_json(state,card/'run_state.json')
    interval,_,_=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),mean_block=b['mean_block_sessions'],resamples=b['draws'],seed=b['seed'],alpha=b['one_sided_alpha'])
    coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1]);means=[float(x.paired_mse_improvement.mean()) for x in halves]
    gates={'coefficient_positive':coefficient>0,'full_held_mean_positive':float(daily.paired_mse_improvement.mean())>0,'both_halves_positive':all(x>0 for x in means),'bootstrap_lower_positive':float(interval['lower_bound'])>0}
    passed=all(gates.values())
    quarter=daily.assign(quarter=daily.signal_date.dt.to_period('Q').astype(str)).groupby('quarter').agg(sessions=('paired_mse_improvement','size'),paired_mean_mse_improvement=('paired_mse_improvement','mean')).reset_index()
    rec.save_objects(artifact_path='predictions',held=daily)
    rec.save_objects(artifact_path='analysis',gates=gates,bootstrap=interval,quarter=quarter,preflight=preflight)
    rec.save_objects(artifact_path='analysis/daily',paired_errors=daily)
    rec.log_metrics(information_gate_pass=int(passed),held_paired_mse_improvement=float(daily.paired_mse_improvement.mean()),bootstrap_lower=float(interval['lower_bound']),native_accounts=0)
    rid,eid=rec.id,str(rec.experiment_id)
result={'status':'diagnostic_information_pass_source_first_public_gap' if passed else 'not_promoted_fixed_Stage1_FT900_export_information_v2_failed','plan_sha256':sha(plan_path),'recorder_id':rid,'experiment_id':eid,'training_mature_rows':len(tr),'held_mature_rows':len(he),'candidate_standardized_export_coefficient':coefficient,'baseline_held_mse':float(np.mean((yhe-predictions['baseline'])**2)),'candidate_held_mse':float(np.mean((yhe-predictions['candidate'])**2)),'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),'held_half_improvements':means,'bootstrap_lower':float(interval['lower_bound']),'gates':gates,'information_gate_pass':passed,'quarterly_information_attribution':quarter.to_dict('records'),'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'historical_first_public_bytes_qualified':False,'classification_continuity_qualified':False,'historical_window_independent':False,'SOXX_goal_achieved':False}
artifacts=tracking/eid/rid/'artifacts'
atomic_json({'recorder_id':rid,'entries':[{'path':str(p.relative_to(card)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
atomic_json(result,card/'result_review.json')
quarter.to_csv(card/'quarterly_information_attribution.csv',index=False)
binding=json.loads((source/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-07');atomic_json(binding,card/'nas_binding.json')
nas=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'result':'result_review.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/eid/rid),'nas_evidence':nas,'model_fits':2,'new_native_accounts':0,'information_gate_pass':passed,'qualified_forward_days':0,'artifact_policy':plan['artifact_policy']}
atomic_json(summary,card/'summary.json')
state.update(status='completed',held_scored_rows=len(he),information_gate_pass=passed);atomic_json(state,card/'run_state.json')
(card/'conclusion.md').write_text(f"# 原Stage1半导体出口增量信息：规范来源固定筛查\n\n原39项Stage1输入与同源发布年龄、2012后分类时期及23日过渡未知控制为B；C只加同份原始FT-900月报半导体出口环比。23个过渡日保留在全日历、候选数值为NaN，由训练域填补器处理；零按结果筛日。Qlib DatasetH训练{len(tr)}行、已复用诊断{len(he)}行；固定两次Ridge拟合和一次63日区块检验，0经济账户。候选标准化系数{coefficient:.8f}，配对MSE改善{result['held_paired_mse_improvement']:.10f}，两半段{means[0]:.10f}/{means[1]:.10f}，预登记下界{result['bootstrap_lower']:.10f}。信息门{'通过' if passed else '失败'}。\n\n决定：`{result['status']}`。FT-900历史首次公开PDF字节未认证；2023–2026是反复使用的诊断历史。即使信息门通过也不授予Stage1 0bp账户或晋升，仍需来源资格、冻结经济设计及独立前向。失败则关闭此固定版本，不改符号、滞后、缺失掩码、年龄、窗口或Ridge。Recorder `{rid}`，NAS SHA `{nas['sha256']}`。\n")
print(json.dumps({'decision':result['status'],'train':len(tr),'held':len(he),'improvement':result['held_paired_mse_improvement'],'lower':result['bootstrap_lower'],'recorder_id':rid}))

## 结论

只有精确计划的独立设计审查通过后才能执行。运行后以 `result_review.json`、中文结论和 Qlib Recorder 为准；无论信息结果如何，历史首发字节、0bp/2bp/5bp 和前向门仍独立。